# Odin-1: LoRA fine-tune of grandpa-brain (O3)

This notebook teaches the **grandpa-brain** base model (**Qwen3 8B**, the same
family as `models/Modelfile.brain`) Grandpa's tools, using the dataset made by
`scripts/odin/make_dataset.py` (O2). It runs on a **free GPU**: Google Colab
(T4) or Kaggle (T4 / P100).

What it does, top to bottom:

1. installs Unsloth;
2. loads `train.jsonl` and `eval.jsonl` (upload them, or attach them as a
   Kaggle dataset);
3. loads Qwen3-8B in 4-bit and adds LoRA adapters;
4. trains (about 30–60 minutes on a T4 for the default dataset);
5. checks a few held-out examples;
6. exports **GGUF Q4_K_M** (about 5 GB) for Ollama.

Then on your PC: `scripts/odin/README.md` → `ollama create grandpa-brain-odin1`.

**Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
**Kaggle:** Settings → Accelerator → **GPU T4 x2** (one is used), Internet **on**, then **Run all**.

In [ ]:
# 1. Install. Takes 2-3 minutes. Restart is not needed.
%pip install -q unsloth
%pip install -q --no-deps "trl>=0.15" datasets
import torch

print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "NONE - switch the runtime to a GPU",
)

In [ ]:
# 2. Settings. Change these only if you know why.
BASE_MODEL = "unsloth/Qwen3-8B"  # same family as models/Modelfile.brain (qwen3:8b)
MAX_SEQ_LENGTH = 4096  # the tool definitions make prompts long
LORA_RANK = 16
EPOCHS = 2
LEARNING_RATE = 2e-4
OUTPUT_NAME = "grandpa-brain-odin1"

In [ ]:
# 3. Find train.jsonl / eval.jsonl, and a place that survives a disconnect.
#    On Colab, Google Drive is mounted first: the dataset and the training
#    checkpoints are kept in MyDrive/grandpa/, so if Colab stops you just press
#    Run all again - no second upload, and training continues where it stopped.
#    (The dataset comes from D:\Grandpa\runtime\odin\ after make_dataset.py.)
import glob
import os
import shutil

SAVE_DIR = "outputs"
try:
    from google.colab import drive

    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/grandpa"
except ImportError:
    if os.path.isdir("/kaggle/working"):
        SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)


def find(name):
    hits = (
        glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        + glob.glob(f"/content/{name}")
        + glob.glob(f"{SAVE_DIR}/{name}")
        + glob.glob(name)
    )
    return hits[0] if hits else None


train_path, eval_path = find("train.jsonl"), find("eval.jsonl")
if not (train_path and eval_path):
    try:
        from google.colab import files

        print("Upload train.jsonl and eval.jsonl (select both).")
        files.upload()
        train_path, eval_path = find("train.jsonl"), find("eval.jsonl")
    except ImportError:
        pass
assert train_path and eval_path, "train.jsonl and eval.jsonl were not found"
# Keep a copy next to the checkpoints, so the next run finds it without asking.
for found in (train_path, eval_path):
    kept = os.path.join(SAVE_DIR, os.path.basename(found))
    if os.path.abspath(found) != os.path.abspath(kept):
        shutil.copy(found, kept)
print(train_path, eval_path, "| checkpoints in", SAVE_DIR)


In [ ]:
# 4. Load the base model in 4-bit and add LoRA adapters.
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=7,
)

In [ ]:
# 5. Turn each example into the model's own chat format, tools included.
import json

from datasets import Dataset


def load(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def render(row):
    tools = row.get("tools") or None
    text = tokenizer.apply_chat_template(
        row["messages"], tools=tools, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}


train_rows, eval_rows = load(train_path), load(eval_path)
train_ds = Dataset.from_list([render(r) for r in train_rows])
eval_ds = Dataset.from_list([render(r) for r in eval_rows])
print(len(train_ds), "train /", len(eval_ds), "eval")
print(train_ds[0]["text"][:1500])

In [ ]:
# 6. Train. On a T4 this is roughly 30-60 minutes for the default dataset.
#    A checkpoint is saved every 4 steps in SAVE_DIR; if a run was cut off,
#    this continues from the last one instead of starting again.
from trl import SFTConfig, SFTTrainer

CHECKPOINTS = os.path.join(SAVE_DIR, "checkpoints")
bf16 = torch.cuda.is_bf16_supported()
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="steps",
        save_steps=4,
        save_total_limit=2,
        optim="adamw_8bit",
        fp16=not bf16,
        bf16=bf16,
        seed=7,
        output_dir=CHECKPOINTS,
        report_to="none",
    ),
)
resume = bool(glob.glob(os.path.join(CHECKPOINTS, "checkpoint-*")))
print("Continuing from the last checkpoint." if resume else "Starting training.")
stats = trainer.train(resume_from_checkpoint=resume)
print(stats)


In [ ]:
# 7. Quick look: does it call the right tool on held-out requests?
#    The full comparison is `grandpa models eval` on your PC (O4).
import re

FastLanguageModel.for_inference(model)


def first_call(row):
    prompt = tokenizer.apply_chat_template(
        row["messages"][:2],
        tools=row.get("tools") or None,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=160, do_sample=False)
    text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1] :], skip_special_tokens=False
    )
    match = re.search(r'"name"\s*:\s*"([^"]+)"', text)
    return (match.group(1) if match else None), text


hits = total = 0
for row in [r for r in eval_rows if r.get("action")][:20]:
    name, text = first_call(row)
    total += 1
    hits += name == row["action"]
    print(
        ("OK " if name == row["action"] else "-- ") + row["messages"][1]["content"],
        "->",
        name,
    )
print(f"tool chosen correctly: {hits}/{total}")

In [ ]:
# 8. Export GGUF Q4_K_M for Ollama (about 5 GB; takes 10-20 minutes).
model.save_pretrained_gguf(OUTPUT_NAME, tokenizer, quantization_method="q4_k_m")
gguf = sorted(glob.glob(f"{OUTPUT_NAME}/*.gguf") + glob.glob(f"{OUTPUT_NAME}*.gguf"))
print(gguf)

In [ ]:
# 9. Get the file to your PC.
#    Colab: it goes to Google Drive (MyDrive/grandpa/), far more reliable than a
#    5 GB browser download. Kaggle: it is in the Output tab (/kaggle/working).
target = gguf[-1]
final = os.path.join(SAVE_DIR, "grandpa-brain-odin1.Q4_K_M.gguf")
shutil.copy(target, final)
print("Saved:", final)


## Next, on your PC

Put the file at `D:\Grandpa\models\odin\grandpa-brain-odin1.Q4_K_M.gguf`, then:

```powershell
cd D:\Grandpa
powershell -ExecutionPolicy Bypass -File scripts\odin\create_odin1.ps1
ollama run grandpa-brain-odin1 "lock the screen"
```

Grandpa keeps using `grandpa-brain` until you choose otherwise. Compare the two
with `grandpa models eval` (O4) before switching.